# ***Quickstart: Querying PDF With Astra and LangChain***

### ***A question-answering demo using Astra DB and LangChain, powered by Vector Search***

#### ***Pre-requisites:***

You need a **_Serverless Cassandra with Vector Search_** database on [Astra DB](https://astra.datastax.com) to run this demo. As outlined in more detail [here](https://docs.datastax.com/en/astra-serverless/docs/vector-search/quickstart.html#_prepare_for_using_your_vector_database), you should get a DB Token with role _Database Administrator_ and copy your Database ID: these connection parameters are needed momentarily.

You also need an [OpenAI API Key](https://cassio.org/start_here/#llm-access) for this demo to work.

#### ***What you will do:***

- Setup: import dependencies, provide secrets, create the LangChain vector store;
- Run a Question-Answering loop retrieving the relevant headlines and having an LLM construct the answer.

In [3]:
import cassio
import langchain
from langchain.vectorstores.cassandra import Cassandra
from langchain.indexes.vectorstore import VectorStoreIndexWrapper
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import HuggingFacePipeline
from langchain_groq import ChatGroq
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter

In [ ]:
# --- Configuration ---
load_dotenv()
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_ID = os.getenv("ASTRA_DB_ID")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [ ]:
# --- Validate environment variables ---
if not ASTRA_DB_APPLICATION_TOKEN or not ASTRA_DB_ID or not GROQ_API_KEY:
    raise ValueError("Missing environment variables. Check your .env file.")

In [5]:
# --- Initialize AstraDB Connection ---
cassio.init(token=ASTRA_DB_APPLICATION_TOKEN, database_id=ASTRA_DB_ID)

In [6]:
# --- Load PDF ---
pdfreader = PdfReader("Budget_Speech.pdf")

In [7]:
raw_text = "".join([page.extract_text() for page in pdfreader.pages if page.extract_text()])

In [8]:
# --- Split Text for Vector Storage ---
text_splitter = CharacterTextSplitter(separator="\n", chunk_size=800, chunk_overlap=200, length_function=len)

In [9]:
texts = text_splitter.split_text(raw_text)

In [10]:
# --- Initialize Vector Store ---
embedding = HuggingFaceEmbeddings()

C:\Users\emon1\AppData\Local\Temp\ipykernel_8516\3643036227.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings()
C:\Users\emon1\AppData\Local\Temp\ipykernel_8516\3643036227.py:2: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding = HuggingFaceEmbeddings()
c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidget

In [11]:
astra_vector_store = Cassandra(embedding=embedding, table_name="qa_mini_demo")

In [12]:
astra_vector_store.add_texts(texts[:50])

['fa32aa362280447f8d2045fef856928b',
 '7e08f1e4553742e0adc66de69728e6fe',
 'd9b1805a3220466d9bbb61f73ee41832',
 'c1d647c3f72942cbb276c1ecbc6ed0f7',
 'e186d728b9e74102a88013784b251540',
 '7d239f7073d6457581bb5d7e5b711c0f',
 '85f2684141f34ae4a4bcaa5302671179',
 '8b6f655ab5f34e5b9f25cd4aebaa4f4c',
 '211ba2a0d25c444c934487b0836ad0c3',
 '31587fbade914e2eb9f1cc2ee756f771',
 '42a824f215ce4f69a3a7b5aab40f47a3',
 '00cceea470954ee0a23a6d44c921034d',
 '32a9057577324bd187e2f6da888654e6',
 '6d7ca3f4e4b14ced8d5e602118864f85',
 'bc349e235f404c8983a25643a4f29db9',
 'de10f1e39ccf428a84c88d1ffae8fe0c',
 '077d18f764ca4f0fb1afc1951b0e6cbb',
 '399071a8088a4175915a37b3f3490cd9',
 'c02045c0ba73458c8ed24d9e71fabbbb',
 'f389ce4a2693472e8acade80f063f751',
 '4d4aca37098544c2ab3da8ffbbabd5b3',
 '5fa3b89683c14f708b8a4fc3d3e1184c',
 'ceb0584e22bc490583e44310d5ba24ef',
 'e0cb55e2072e447bb1a95456b4ca368c',
 '276024bcab014c659a7d3cbbb70cc421',
 '5d296c24187d4d0381164c4ef6da39e0',
 'f58441fe16b8435b9ecb18c350f7128c',
 

In [13]:
astra_vector_index = VectorStoreIndexWrapper(vectorstore=astra_vector_store)

In [18]:
# --- Initialize LLM using Groq ---
llm = ChatGroq(model="gemma2-9b-it", groq_api_key=GROQ_API_KEY)

In [19]:
# --- Interactive Q&A Loop ---
first_question = True
while True:
    query_text = input("\nEnter your question (or type 'quit' to exit): ").strip() if first_question else input("\nWhat's your next question (or type 'quit' to exit): ").strip()
    if query_text.lower() == "quit":
        break
    first_question = False
    print(f"\nQUESTION: {query_text}")
    answer = astra_vector_index.query(query_text, llm=llm).strip()
    print(f"ANSWER: {answer}\n")
    print("FIRST DOCUMENTS BY RELEVANCE:")
    for doc, score in astra_vector_store.similarity_search_with_score(query_text, k=4):
        print(f"    [{score:.4f}] {doc.page_content[:84]} ...")


QUESTION: what is the budget of 2025
ANSWER: This document does not contain the budget for 2025.  It outlines the proposed budget for FY2024-25.

FIRST DOCUMENTS BY RELEVANCE:
    [0.8201] Chapter  Five 
Proposed Budget for FY2024 -25  
The Proposed Budget of FY 2024-25; R ...
    [0.7408] 6 Sectoral Allocation in Budget  178 
7 Ministry/Division -wise Budget Allocation  1 ...
    [0.7161] The Success of Bangladesh in the Last Decade and a Half 
Under the Strong Leadership ...
    [0.7155] e-GP and iBAS++, Welfare of pensioners, Development of 
government cash management,  ...

QUESTION: what is Establishment of Good governance?
ANSWER: While this text mentions many development goals, it doesn't explicitly define "Establishment of Good governance".

FIRST DOCUMENTS BY RELEVANCE:
    [0.7104] education,  scientific  research  and innovation  
• Providing  necessary  support   ...
    [0.7043] rate to 3 percent  by 2041.  
• Ensuring  conducive  environment  for setting  up in ...
    [

KeyboardInterrupt: Interrupted by user